# DBSCAN clustering

Explore density-based student grouping and compare its noise handling with K-means.

**Dataset:** `data/student_data.csv` (395 student records, 33 columns). **Target:** final grade `G3` (0–20). For classification labs, pass means `G3 ≥ 10`. Earlier grades `G1` and `G2` are included as predictors; interpret results as prediction after those earlier grades are known. The split is fixed with random seed 42 for reproducibility.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATHS = [Path("data/student_data.csv"), Path("../data/student_data.csv")]
DATA_PATH = next((p for p in DATA_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place student_data.csv in the repository's data/ folder.")
df = pd.read_csv(DATA_PATH)
TARGET = "G3"  # final-period grade (0–20)
PASS_THRESHOLD = 10
numeric_features = df.select_dtypes(include=np.number).columns.drop(TARGET).tolist()
categorical_features = df.select_dtypes(exclude=np.number).columns.tolist()
feature_cols = numeric_features + categorical_features
X = df[feature_cols]
y_reg = df[TARGET]
y_class = (df[TARGET] >= PASS_THRESHOLD).astype(int)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

cluster_features=["age","studytime","failures","absences","G1","G2"]
X_scaled=StandardScaler().fit_transform(df[cluster_features])
k_dist=NearestNeighbors(n_neighbors=10).fit(X_scaled).kneighbors(X_scaled)[0][:,-1]
plt.plot(np.sort(k_dist)); plt.ylabel("Distance to 10th neighbor"); plt.xlabel("Sorted student index"); plt.title("DBSCAN k-distance plot"); plt.show()
# Treat eps as an experiment parameter; inspect the plot and adjust if needed.
eps_value=1.8
db=DBSCAN(eps=eps_value,min_samples=10).fit_predict(X_scaled)
cluster_ids=set(db)-{-1}
print("DBSCAN clusters:",len(cluster_ids),"| noise points:",int((db==-1).sum()),f"({(db==-1).mean():.1%})")
if len(cluster_ids)>1:
    print("DBSCAN silhouette (excluding noise):",round(silhouette_score(X_scaled[db!=-1],db[db!=-1]),3))
else:
    print("Silhouette unavailable: DBSCAN found fewer than two non-noise clusters.")
km=KMeans(n_clusters=max(2,len(cluster_ids)),n_init=20,random_state=42).fit_predict(X_scaled)
print("K-means silhouette:",round(silhouette_score(X_scaled,km),3))
fig,axes=plt.subplots(1,2,figsize=(12,5))
axes[0].scatter(df.G1,df.G2,c=db,cmap="tab10",s=25); axes[0].set_title("DBSCAN (noise = -1)")
axes[1].scatter(df.G1,df.G2,c=km,cmap="tab10",s=25); axes[1].set_title("K-means comparison")
for ax in axes: ax.set_xlabel("G1"); ax.set_ylabel("G2")
plt.tight_layout(); plt.show()
